# Explaining a supplied fact with Qwen3.5
Run every cell in order in a CPU notebook. We investigate a fixed red-versus-blue
score, then ask whether input edits and internal replacements change it.

In [ ]:
import importlib.metadata as metadata
import importlib.util
import subprocess
import sys

PINS = {'transformers': '5.22.1', 'captum': '0.9.0',
        'matplotlib': '3.11.2', 'psutil': '7.2.2'}
# Colab may preload matplotlib; retain it to avoid mixing loaded and installed versions.
missing = []
for package, version in PINS.items():
    try:
        if metadata.version(package) != version and package != 'matplotlib':
            missing.append(f'{package}=={version}')
    except metadata.PackageNotFoundError:
        missing.append(f'{package}=={version}')
if importlib.util.find_spec('torch') is None:
    missing.append('torch==2.15.0')
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

## Loading the text model
This checkpoint also has a vision branch, but the class below loads its language
model. Its 24 blocks mix gated DeltaNet operations and full attention. We keep
the pretrained parameters fixed and use ordinary PyTorch gradients and hooks.

In [ ]:
import os
import json
import time
import psutil
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from captum.attr import IntegratedGradients
from transformers import AutoTokenizer, Qwen3_5ForCausalLM

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
torch.manual_seed(17)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device('cpu')
OUT = Path(os.environ.get('CH17_LANGUAGE_OUT', 'outputs/language'))
OUT.mkdir(parents=True, exist_ok=True)
STARTED = time.perf_counter()
MODEL_ID = 'Qwen/Qwen3.5-0.8B'
MODEL_REVISION = '2fc06364715b967f1860aea9cf38778875588b17'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = Qwen3_5ForCausalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.float32,
    attn_implementation='eager').to(DEVICE).eval()
model.requires_grad_(False)
print(f'Frozen language parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Blocks: {len(model.model.layers)}; device: {DEVICE}')

## Defining exactly which output we explain
We compare the next-token scores for `red` and `blue`. The difference is a logit
margin, not the probability that the whole answer is correct. Tokenization is
checked, rather than assuming a word occupies one token.

In [ ]:
CANDIDATES = ('red', 'blue')
candidate_tokens = [tokenizer.encode(w, add_special_tokens=False) for w in CANDIDATES]
assert all(len(ids) == 1 for ids in candidate_tokens)
RED_ID, BLUE_ID = [ids[0] for ids in candidate_tokens]

def make_prompt(note, person='Mira'):
    question = ("Read the note. Answer with one colour only.\n"
                f"Note: {note}\nQuestion: What colour is {person}'s badge?")
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': question}], tokenize=False,
        add_generation_prompt=True, enable_thinking=False)

def score_ids(input_ids):
    mask = torch.ones_like(input_ids)
    logits = model(input_ids=input_ids, attention_mask=mask,
                   use_cache=False).logits[:, -1]
    return logits[:, RED_ID] - logits[:, BLUE_ID]

def score_note(note, person='Mira'):
    ids = tokenizer(make_prompt(note, person), return_tensors='pt').input_ids.to(DEVICE)
    with torch.no_grad():
        return score_ids(ids).item()

NOTE = "Mira's badge is red. Joel's badge is green."
CORRUPTED_NOTE = "Mira's badge is blue. Joel's badge is green."
clean_score = score_note(NOTE)
corrupt_score = score_note(CORRUPTED_NOTE)
def short_answer(note):
    inputs = tokenizer(make_prompt(note), return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        generated = model.generate(**inputs, max_new_tokens=4, do_sample=False)
    return tokenizer.decode(generated[0, inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

clean_answer = short_answer(NOTE)
corrupt_answer = short_answer(CORRUPTED_NOTE)
print(f'Answers: original={clean_answer!r}, changed={corrupt_answer!r}')
print(f'Original note: {clean_score:+.3f}')
print(f'Changed target colour: {corrupt_score:+.3f}')
assert clean_score > 0 and corrupt_score < 0

# Keep both facts and the question the same; change only their order.
joel_first_note = "Joel's badge is green. Mira's badge is red."
joel_first_answer = short_answer(joel_first_note)
joel_first_score = score_note(joel_first_note)
print(f'Joel first: answer={joel_first_answer!r}; red-minus-blue score={joel_first_score:+.3f}')

## Editing one word at a time
Each row substitutes `unknown` for one specified word in the note. We leave the
question alone. Removing information and introducing an unfamiliar input are
both possible effects of such an edit, so we report the edited text too.

In [ ]:
words = ['Mira', 'red', 'Joel', 'green']
word_edits = []
for word in words:
    edited = NOTE.replace(word, 'unknown', 1)
    edited_score = score_note(edited)
    word_edits.append({'word': word, 'note': edited, 'score': edited_score,
                       'drop': clean_score - edited_score})
    print(f'{word:>5} -> unknown: score {edited_score:+.3f}; drop {clean_score-edited_score:+.3f}')

## Integrated Gradients needs a reference
Integer token identifiers are not differentiable. We work with their embedding
vectors, holding the question and chat formatting fixed. Our reference is a
real alternative note: Mira has blue, Joel has red. The attribution therefore
explains the score difference between these two notes, not absolute importance.

In [ ]:
prompt = make_prompt(NOTE)
encoded = tokenizer(prompt, return_tensors='pt', return_offsets_mapping=True)
offsets = encoded.pop('offset_mapping')[0].tolist()
clean_ids = encoded.input_ids.to(DEVICE)
REFERENCE_NOTE = "Mira's badge is blue. Joel's badge is red."
reference_ids = tokenizer(make_prompt(REFERENCE_NOTE), return_tensors='pt').input_ids.to(DEVICE)
assert clean_ids.shape == reference_ids.shape
embeddings = model.get_input_embeddings()(clean_ids).detach()
reference = model.get_input_embeddings()(reference_ids).detach()

def score_embeddings(values):
    mask = torch.ones(values.shape[:2], dtype=torch.long, device=values.device)
    logits = model(inputs_embeds=values, attention_mask=mask,
                   use_cache=False).logits[:, -1]
    return logits[:, RED_ID] - logits[:, BLUE_ID]

with torch.no_grad():
    torch.testing.assert_close(score_embeddings(embeddings), score_ids(clean_ids))
    reference_score = score_embeddings(reference).item()
ig_started = time.perf_counter()
attributions, delta = IntegratedGradients(score_embeddings).attribute(
    embeddings, baselines=reference, n_steps=32, internal_batch_size=1,
    return_convergence_delta=True)
token_attribution = attributions.sum(dim=-1)[0].detach().cpu().numpy()
ig_error = float(delta.item())
score_change = clean_score - reference_score
print(f'Endpoint score difference: {score_change:+.4f}')
print(f'Attribution sum: {token_attribution.sum():+.4f}; numerical residual: {ig_error:+.4f}')
assert np.isfinite(token_attribution).all()
assert abs(ig_error) < max(0.10, 0.02 * abs(score_change)), 'Increase integration steps before interpreting this result.'
ig_seconds = time.perf_counter() - ig_started
note_start = prompt.index(NOTE)
word_ig = []
for word in words:
    a = note_start + NOTE.index(word); b = a + len(word)
    indices = [i for i, (start, end) in enumerate(offsets) if start < b and end > a]
    word_ig.append(float(token_attribution[indices].sum()))
print('Selected-word attributions:',dict(zip(words, np.round(word_ig, 4))))

In [ ]:
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 9.5})
fig, axes = plt.subplots(1, 2, figsize=(5.4, 3.2), layout='constrained')
for ax, values, title in zip(axes, [word_ig, [r['drop'] for r in word_edits]],
                            ['Integrated Gradients', 'One-word edits']):
    ax.barh(words, values, color=['#147d77' if v >= 0 else '#bd643b' for v in values])
    ax.axvline(0, color='#555555', linewidth=.7)
    ax.invert_yaxis(); ax.set_title(title, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
axes[0].set_xlabel('Attribution relative to\nalternative note')
axes[1].set_xlabel('Score drop after\nreplacement with unknown')
fig.suptitle('Two methods ask different questions', fontweight='bold')
fig.savefig(OUT/'language_attribution.png',dpi=240); fig.savefig(OUT/'language_attribution.svg')
plt.show()

## Replacing an internal activation
Cache the actual outputs of selected blocks on the clean run. Then replace one
token's hidden vector on the corrupted run, using a temporary forward hook.
The hook is always removed. We do not retrain or modify any model parameter.

In [ ]:
LAYERS = [0, 3, 7, 11, 15, 19, 23]
clean_states = {}
handles = []
def cache_output(index):
    def hook(module, args, output):
        state = output[0] if isinstance(output, tuple) else output
        clean_states[index] = state.detach().clone()
    return hook
try:
    for index in LAYERS:
        handles.append(model.model.layers[index].register_forward_hook(cache_output(index)))
    with torch.no_grad():
        captured_score = score_ids(clean_ids)
finally:
    for handle in handles:
        handle.remove()
torch.testing.assert_close(captured_score, torch.tensor([clean_score],device=DEVICE))
corrupt_ids = tokenizer(make_prompt(CORRUPTED_NOTE), return_tensors='pt').input_ids.to(DEVICE)
assert corrupt_ids.shape == clean_ids.shape
changed_positions = (clean_ids[0] != corrupt_ids[0]).nonzero().flatten().tolist()
assert len(changed_positions) == 1
colour_position = changed_positions[0]
name_position = next(i for i, (a,b) in enumerate(offsets) if a <= note_start < b)
question_name_start = prompt.index("Mira's badge?", note_start+len(NOTE))
question_position = next(i for i,(a,b) in enumerate(offsets) if a <= question_name_start < b)
POSITIONS = [name_position, colour_position, question_position, clean_ids.shape[1]-1]
POSITION_NAMES = ['Name before the edit', 'Changed colour', 'Name in the question', 'Answer position']

def patched_score(input_ids, layer, position, source_states):
    def replace(module, args, output):
        state = output[0] if isinstance(output, tuple) else output
        changed = state.clone()
        changed[:, position] = source_states[layer][:, position]
        return (changed, *output[1:]) if isinstance(output, tuple) else changed
    handle = model.model.layers[layer].register_forward_hook(replace)
    try:
        with torch.no_grad():
            return score_ids(input_ids).item()
    finally:
        handle.remove()

In [ ]:
patch_started = time.perf_counter()
patch_scores = np.array([[patched_score(corrupt_ids, layer, position, clean_states)
                         for layer in LAYERS] for position in POSITIONS])
assert np.allclose(patch_scores[0], corrupt_score, atol=2e-4)
self_patch = patched_score(clean_ids, LAYERS[2], colour_position, clean_states)
assert abs(self_patch-clean_score) < 2e-4
recovery = (patch_scores-corrupt_score)/(clean_score-corrupt_score)
fig, ax = plt.subplots(figsize=(5.4, 3.4),layout='constrained')
image = ax.imshow(recovery, cmap='RdBu_r', vmin=-.15, vmax=1.15, aspect='auto')
ax.set_xticks(range(len(LAYERS)),[str(i+1) for i in LAYERS]);ax.set_xlabel('Replacement after block')
ax.set_yticks(range(len(POSITIONS)),POSITION_NAMES)
for row in range(len(POSITIONS)):
    for col in range(len(LAYERS)):
        value=recovery[row,col];ax.text(col,row,f'{value:.2f}',ha='center',va='center',fontsize=9,
                                      color='white' if value<.15 or value>.85 else 'black')
fig.colorbar(image,ax=ax,label='Recovery of the original score difference',shrink=.85)
fig.suptitle('Where does a replacement recover the score?',fontweight='bold',fontsize=11)
fig.savefig(OUT/'activation_patching.png',dpi=240);fig.savefig(OUT/'activation_patching.svg');plt.show()
patch_seconds = time.perf_counter()-patch_started
print(f'Unpatched: {corrupt_score:+.3f}; original: {clean_score:+.3f}')
print('Colour-token recovery:',np.round(recovery[1],3).tolist())
print(f'Same-input replacement agrees: {abs(self_patch-clean_score):.2g}')

## Checking other prompts
These checks were specified before inspecting the attribution plots. They test
the input behaviour, not a claim that the same internal mechanism handles every
name, language, or context. A few cases cannot establish general reliability.

In [ ]:
held_out = []
for person, other in [('Alice','Bob'),('Nora','Liam')]:
    for colour in ['red','blue']:
        note = f"{person}'s badge is {colour}. {other}'s badge is green."
        value = score_note(note,person)
        held_out.append({'person':person,'colour':colour,'score':value})
        print(f'{person}, {colour}: {value:+.3f}')
        assert (value>0) == (colour=='red')
unrelated_note = "Mira's badge is red. Joel's badge is yellow."
unrelated_score = score_note(unrelated_note)
print(f'Changed only Joel\'s colour: {unrelated_score:+.3f}')
assert unrelated_score > 0

## Optional: repeating the swap with Joel’s badge changed
Run the preceding cells first, then set `TRY_YELLOW_BADGE` to `True` and run
this cell. Joel has yellow in both notes; only Mira’s red/blue colour changes
between them. We try the colour swap after the first and last blocks.

In [ ]:
TRY_YELLOW_BADGE = False
if TRY_YELLOW_BADGE:
    def yellow_ids(note):
        return tokenizer(
            make_prompt(note.replace('green', 'yellow')),
            return_tensors='pt',
        ).input_ids.to(DEVICE)

    yellow_red_ids = yellow_ids(NOTE)
    yellow_blue_ids = yellow_ids(CORRUPTED_NOTE)
    assert yellow_red_ids.shape == yellow_blue_ids.shape
    changed = (yellow_red_ids[0] != yellow_blue_ids[0]).nonzero().flatten().tolist()
    assert len(changed) == 1
    yellow_colour_position = changed[0]

    # Capture separately so the optional test leaves the main experiment intact.
    yellow_states = {}
    def save_yellow_state(layer):
        def capture(module, args, output):
            state = output[0] if isinstance(output, tuple) else output
            yellow_states[layer] = state.detach().clone()
        return capture

    for layer in (LAYERS[0], LAYERS[-1]):
        handle = model.model.layers[layer].register_forward_hook(save_yellow_state(layer))
        try:
            with torch.no_grad():
                captured = score_ids(yellow_red_ids).item()
        finally:
            handle.remove()
        with torch.no_grad():
            red_score = score_ids(yellow_red_ids).item()
            blue_score = score_ids(yellow_blue_ids).item()
        assert abs(captured - red_score) < 2e-4
        same = patched_score(yellow_red_ids, layer, yellow_colour_position, yellow_states)
        assert abs(same - red_score) < 2e-4
        patched = patched_score(yellow_blue_ids, layer, yellow_colour_position, yellow_states)
        print(f'Joel has yellow, block {layer + 1}: blue note {blue_score:+.3f}, '
              f'after swap {patched:+.3f}, red note {red_score:+.3f}')

## Saving the evidence
Store numerical results and plots. The memory figure is the process high-water
resident memory reported by the operating system, not a GPU allocation measure.
The weights remain fixed; no weights, private data, or generated traces are committed.

In [ ]:
try:
    import resource
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    peak_bytes = int(peak if sys.platform=='darwin' else peak*1024)
except ImportError:
    peak_bytes = getattr(psutil.Process().memory_info(), 'peak_wset', None)
results = {'model':MODEL_ID,'model_revision':MODEL_REVISION,'device':str(DEVICE),'dtype':'float32',
           'parameters':sum(p.numel() for p in model.parameters()),'blocks':len(model.model.layers),
           'clean_answer':clean_answer,'corrupt_answer':corrupt_answer,'note':NOTE,'corrupted_note':CORRUPTED_NOTE,'reference_note':REFERENCE_NOTE,
           'candidate_tokens':candidate_tokens,'clean_score':clean_score,'corrupt_score':corrupt_score,
           'joel_first_note':joel_first_note,'joel_first_answer':joel_first_answer,'joel_first_score':joel_first_score,
           'reference_score':reference_score,'word_edits':word_edits,'word_ig':dict(zip(words,word_ig)),
           'ig_steps':32,'ig_sum':float(token_attribution.sum()),'ig_delta':ig_error,'ig_seconds':ig_seconds,
           'layers':LAYERS,'positions':POSITIONS,'position_names':POSITION_NAMES,
           'patch_scores':patch_scores.tolist(),'recovery':recovery.tolist(),'patch_seconds':patch_seconds,
           'self_patch':self_patch,'held_out':held_out,'unrelated_score':unrelated_score,
           'process_peak_rss_bytes':peak_bytes,'seconds_including_model_loading':time.perf_counter()-STARTED,
           'versions':{name:metadata.version(name) for name in ['torch',*PINS]}}
(OUT/'results.json').write_text(json.dumps(results,indent=2)+'\n')
np.savez(OUT/'token_attributions.npz',token_ids=clean_ids.cpu().numpy(),attributions=token_attribution)
print(f'Saved results to {OUT}; process peak memory bytes: {peak_bytes}')